In [1]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

block_size = 8
batch_size = 4

cuda


In [2]:
from google.colab import drive  #type:ignore
import os

# Chỉ mount nếu chưa thấy thư mục drive xuất hiện
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

file_path = '/content/drive/MyDrive/wizard_of_oz.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Độ dài của file: {len(text)} ký tự")
print(text[:100])

chars = sorted(set(text))
print(chars)

Độ dài của file: 232307 ký tự
 DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, 
['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
str_to_int={ch:i for i,ch in enumerate(chars)}
int_to_str={i:ch for i,ch in enumerate(chars)}


encode = lambda s: [str_to_int[c] for c in s]
decode = lambda l: "".join([int_to_str[i] for i in l])

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.8 * len(data))   
train_data = data[:n]       
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    x,y = x.to(device), y.to(device)
    
    return x,y

x,y = get_batch('train')
print('input')
print(x)
print('target')
print(y)

tensor([75956, 61152, 82295, 34687])
input
tensor([[62, 67, 60,  1, 73, 61, 58, 66],
        [62, 67, 58,  1, 73, 62, 56, 64],
        [67,  5, 73,  1, 64, 67, 68, 76],
        [ 0, 57, 68, 76, 67,  1, 74, 69]], device='cuda:0')
target
tensor([[67, 60,  1, 73, 61, 58, 66,  1],
        [67, 58,  1, 73, 62, 56, 64, 58],
        [ 5, 73,  1, 64, 67, 68, 76,  1],
        [57, 68, 76, 67,  1, 74, 69, 68]], device='cuda:0')


In [6]:
x=train_data[:block_size]
y=train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]   
    print(f"when input is {context} the target: {target}")

when input is tensor([1]) the target: 28
when input is tensor([ 1, 28]) the target: 39
when input is tensor([ 1, 28, 39]) the target: 42
when input is tensor([ 1, 28, 39, 42]) the target: 39
when input is tensor([ 1, 28, 39, 42, 39]) the target: 44
when input is tensor([ 1, 28, 39, 42, 39, 44]) the target: 32
when input is tensor([ 1, 28, 39, 42, 39, 44, 32]) the target: 49
when input is tensor([ 1, 28, 39, 42, 39, 44, 32, 49]) the target: 1


In [ ]:
vocab_size = len(chars)

import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets):
        logits = self.token_embedding_table(index)
        return logits